## CNN + LSTM Activity Classification of Public Nusaince Analysis 


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [ ]:
num_clips = 100
sequence_length = 16
feature_size = 512

X = np.random.randn(
    num_clips,
    sequence_length,
    feature_size
)

print("Feature shape:", X.shape)

In [ ]:
y = np.random.randint(0, 3, num_clips)

print("Labels:", y.shape)

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
class CNNLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        last_hidden = hidden[-1]

        prediction = self.fc(last_hidden)

        return prediction

In [ ]:
input_size = 512
hidden_size = 128
num_classes = 3

model = CNNLSTM(
    input_size,
    hidden_size,
    num_classes
)

model = model.to(device)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
X_train = X_train.to(device)
X_test = X_test.to(device)

y_train = y_train.to(device)
y_test = y_test.to(device)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    output = model(X_train)

    loss = criterion(output, y_train)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 2 == 0:
        print(
            "Epoch:",
            epoch + 1,
            "Loss:",
            round(loss.item(), 4)
        )

In [ ]:
model.eval()

with torch.no_grad():

    output = model(X_test)

    predictions = torch.argmax(
        output,
        dim=1
    )

accuracy = accuracy_score(
    y_test.cpu(),
    predictions.cpu()
)

print("Test Accuracy:", accuracy)

In [ ]:
print(
    classification_report(
        y_test.cpu(),
        predictions.cpu()
    )
)

In [ ]:
features

In [ ]:
sample_clip = features.float()

sample_clip = sample_clip.unsqueeze(0)

sample_clip = sample_clip.to(device)

print("Input shape:", sample_clip.shape)

In [ ]:
model.eval()

with torch.no_grad():

    output = model(sample_clip)

    prediction = torch.argmax(
        output,
        dim=1
    )

print("Predicted Class:", prediction.item())

In [ ]:
class_names = [
    "Abuse",
    "Assault",
    "Normal"
]

In [ ]:
predicted_class = prediction.item()

print(
    "Predicted Activity:",
    class_names[predicted_class]
)